# 4モデルアンサンブル（ベスト構成）
yolo26m + RT-DETR + RF-DETR + yolo26l → WBF → 提出

In [ ]:
import os
os.chdir(r'C:\compe')

import json
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image as PILImage
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRBase
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

with open(r'C:\compe\test_dataset.json') as f:
    test_data = json.load(f)

cat_ids = sorted([c['id'] for c in test_data['categories']])
yolo_to_category = cat_ids

fname_to_id    = {img['file_name']: img['id'] for img in test_data['images']}
img_id_to_info = {img['id']: img for img in test_data['images']}

TEST_DIR   = r'C:\compe\images\test'
test_files = sorted(os.listdir(TEST_DIR))
print(f'テスト画像数: {len(test_files)}')

In [ ]:
# モデル読み込み
model_yolom = YOLO(r'C:\compe\runs\full2\weights\yolom best.pt')
print('yolo26m loaded')

model_yolol = YOLO(r'C:\Users\tamkn\Downloads\best.pt')
print('yolo26l loaded')

model_rtdetr = RTDETR(r'C:\compe\runs\rtdetr4\weights\best.pt')
print('RT-DETR loaded')

model_rfdetr = RFDETRBase(
    pretrain_weights=r'C:\compe\runs\rfdetr\checkpoint_best_total.pth',
    num_classes=32
)
model_rfdetr.optimize_for_inference()
print('RF-DETR loaded')

In [ ]:
# 推論設定
CONF             = 0.05   # 低信頼度を除外
IOU              = 0.6
MAX_DET          = 1000
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.02   # 低信頼度を除外
RFDETR_THR       = 0.15   # RF-DETRも高conf

# yolo26m : RT-DETR : RF-DETR : yolo26l
WEIGHTS = [0.5, 0.5, 3.0, 2.0]

TTA_PATTERNS = [
    (False, 1024),
    (True,  1024),
    (False,  896),
]

In [ ]:
def predict_yolo_tta(model, img_path, conf, iou):
    img_orig = PILImage.open(img_path).convert('RGB')
    all_boxes, all_scores, all_labels = [], [], []

    for do_flip, sz in TTA_PATTERNS:
        img = img_orig.copy()
        if do_flip:
            img = TF.hflip(img)

        result = model.predict(
            source=img, conf=conf, iou=iou,
            max_det=MAX_DET, imgsz=sz,
            verbose=False, half=True,
        )[0]

        if len(result.boxes) == 0:
            continue

        boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
        scores = result.boxes.conf.cpu().numpy().tolist()
        labels = result.boxes.cls.cpu().numpy().astype(int).tolist()

        if do_flip:
            boxes = [[1-x2, y1, 1-x1, y2] for x1, y1, x2, y2 in boxes]

        boxes = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)

    if not all_boxes:
        return [], [], []

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=[1.0] * len(all_boxes),
        iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP_BOX_THR,
    )
    return boxes_f.tolist(), scores_f.tolist(), labels_f.tolist()


def predict_rfdetr(model, img_path, w, h, threshold):
    img = PILImage.open(img_path).convert('RGB')
    result = model.predict(img, threshold=threshold)
    if len(result) == 0:
        return [], [], []
    valid = result.class_id < 32
    if not valid.any():
        return [], [], []
    boxes  = result.xyxy[valid].astype(float)
    boxes[:, [0, 2]] /= w
    boxes[:, [1, 3]] /= h
    boxes  = np.clip(boxes, 0, 1).tolist()
    scores = result.confidence[valid].tolist()
    labels = result.class_id[valid].tolist()
    return boxes, scores, labels

In [ ]:
rows = []

for fname in tqdm(test_files):
    img_path = os.path.join(TEST_DIR, fname)
    image_id = fname_to_id.get(fname)
    if image_id is None:
        stem = Path(fname).stem
        for key in fname_to_id:
            if Path(key).stem == stem:
                image_id = fname_to_id[key]
                break
    if image_id is None:
        continue

    w = img_id_to_info[image_id]['width']
    h = img_id_to_info[image_id]['height']

    boxes_m,  scores_m,  labels_m  = predict_yolo_tta(model_yolom, img_path, CONF, IOU)
    boxes_l,  scores_l,  labels_l  = predict_yolo_tta(model_yolol, img_path, CONF, IOU)
    boxes_r,  scores_r,  labels_r  = predict_yolo_tta(model_rtdetr, img_path, CONF, IOU)
    boxes_rf, scores_rf, labels_rf = predict_rfdetr(model_rfdetr, img_path, w, h, RFDETR_THR)

    boxes_list, scores_list, labels_list, weights_used = [], [], [], []

    if len(boxes_m) > 0:
        boxes_list.append(boxes_m);  scores_list.append(scores_m);  labels_list.append(labels_m);  weights_used.append(WEIGHTS[0])
    if len(boxes_r) > 0:
        boxes_list.append(boxes_r);  scores_list.append(scores_r);  labels_list.append(labels_r);  weights_used.append(WEIGHTS[1])
    if len(boxes_rf) > 0:
        boxes_list.append(boxes_rf); scores_list.append(scores_rf); labels_list.append(labels_rf); weights_used.append(WEIGHTS[2])
    if len(boxes_l) > 0:
        boxes_list.append(boxes_l);  scores_list.append(scores_l);  labels_list.append(labels_l);  weights_used.append(WEIGHTS[3])

    if not boxes_list:
        continue

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights_used,
        iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP_BOX_THR,
    )

    for box, score, label in zip(boxes_f, scores_f, labels_f):
        x1, y1, x2, y2 = box
        rows.append({
            'image_id':    image_id,
            'category_id': yolo_to_category[int(label)],
            'bbox_x':      x1 * w,
            'bbox_y':      y1 * h,
            'bbox_width':  (x2 - x1) * w,
            'bbox_height': (y2 - y1) * h,
            'score':       float(score),
        })

print(f'予測数: {len(rows)}')
print(f'1画像あたり平均: {len(rows)/len(test_files):.1f}box')

In [ ]:
# 可視化確認
import matplotlib.pyplot as plt
import matplotlib.patches as patches

df_check = pd.DataFrame(rows)
sample_id = df_check['image_id'].iloc[0]
sample = df_check[df_check['image_id'] == sample_id]
img_info = img_id_to_info[sample_id]

fname = [k for k, v in fname_to_id.items() if v == sample_id][0]
img_path = os.path.join(TEST_DIR, Path(fname).stem + '.jpg')
img = PILImage.open(img_path)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(img)
cat_names = {c['id']: c['name'] for c in test_data['categories']}

for _, row in sample.iterrows():
    x, y, w, h = row['bbox_x'], row['bbox_y'], row['bbox_width'], row['bbox_height']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y-5, f"{cat_names.get(row['category_id'], '?')} {row['score']:.2f}",
            color='red', fontsize=8, backgroundcolor='white')

plt.title(f'image_id={sample_id} | {len(sample)}個検出')
plt.show()

In [ ]:
# 提出ファイル作成（別名で保存）
from datetime import datetime
today = datetime.now().strftime('%Y%m%d_%H%M')

submission = pd.DataFrame(rows)
submission['annotation_id'] = np.arange(len(submission))
submission = submission[[
    'annotation_id', 'image_id', 'category_id',
    'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score'
]]
submission['score'] = submission['score'].clip(0, 1)

# 日付付きで保存（上書き防止）
save_path = rf'C:\compe\submission_{today}.csv'
submission.to_csv(save_path, index=False)
submission.to_csv(r'C:\compe\submission.csv', index=False)
print(f'完了！{len(submission)}行')
print(f'保存先: {save_path}')
print(submission.head())